In [ ]:
import os  # OS 경로/폴더 생성 등 시스템 기능 사용
import torch  # PyTorch 메인 패키지
import torch.nn as nn  # 신경망 모듈(레이어, 손실 등)
from sklearn.model_selection import StratifiedKFold  # 계층적 K-Fold 분할
from tqdm import tqdm  # 진행률 표시 바
from torchvision import models, transforms  # 사전학습 모델/이미지 변환
from torch.utils.data import Dataset, DataLoader  # PyTorch 데이터셋/로더
from datasets import load_dataset, ClassLabel, DatasetDict  # Hugging Face Datasets 유틸
import numpy as np  # 수치 계산
from torch.amp import autocast, GradScaler  # 혼합정밀 자동 캐스트/스케일러
import math  # 수학 유틸
import time  # 시간 측정
from typing import Optional, Tuple, Dict  # 타입 힌트
import torch.nn.functional as F  # 함수형 API (loss/activation 등)
from sklearn.metrics import f1_score, classification_report, confusion_matrix, balanced_accuracy_score, top_k_accuracy_score  # 평가 지표
from datasets import load_from_disk  # HF dataset 디스크 로드(현재 코드에선 미사용)
from torch.autograd import Variable  # 오토그라드 Variable(현 PyTorch에선 텐서와 동일, 미사용)
from collections import Counter  # 라벨 카운트
import matplotlib.pyplot as plt

import json, os  # JSON 입출력/OS 유틸(중복 import but harmless)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # 사용 디바이스 선택(CUDA 우선)

train_transform = transforms.Compose([  # 학습용 이미지 증강/전처리 파이프라인
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0), ratio=(3/4, 4/3)),  # 랜덤 크롭+리사이즈
    transforms.RandomHorizontalFlip(p=0.5),  # 좌우 반전
    transforms.RandomRotation(degrees=15),  # 회전
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15, hue=0.02),  # 색상/밝기/대비 변형
    transforms.ToTensor(),  # 텐서 변환(CHW, [0,1])
    transforms.Normalize([0.485,0.456,0.406],  # ImageNet 평균/표준편차 정규화
                         [0.229,0.224,0.225]),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.1), ratio=(0.3, 3.3))  # 랜덤 지우기(regularization)
])

val_transform = transforms.Compose([  # 검증용(약한) 전처리
    transforms.Resize(256),  # 리사이즈
    transforms.CenterCrop(224),  # 중앙 크롭
    transforms.ToTensor(),  # 텐서 변환
    transforms.Normalize([0.485,0.456,0.406],  # 정규화
                         [0.229,0.224,0.225])
])


def prepare_dataset():  # 데이터셋 로드/클린업/스플릿/라벨정리 함수
    dataset = load_dataset("Densu341/Fresh-rotten-fruit")  # HF 허브에서 데이터 로드

    # 1) 라벨 제거
    remove_labels = [18, 20, 16, 13, 2, 5, 7, 9]  # 제거할 라벨 인덱스 목록
    labels = np.array(dataset["train"]["label"])  # 원본 라벨 배열
    mask = ~np.isin(labels, remove_labels)  # 제거할 라벨 제외 마스크
    clean = dataset["train"].select(np.where(mask)[0])  # 필터링된 서브셋

    # 2) split (결정적)
    split = clean.train_test_split(test_size=0.2, seed=42)  # train/val 80:20 고정 분할
    train_ds, val_ds = split["train"], split["test"]  # 분할 결과

    # 3) 라벨 재매핑 (이름 기준으로 0..C-1)
    uniq = sorted(set(train_ds["label"]) | set(val_ds["label"]))  # 사용되는 라벨 인덱스 집합
    names = [train_ds.features["label"].int2str(i) for i in uniq]  # 라벨 이름 리스트
    new_lbl = ClassLabel(num_classes=len(names), names=names)  # 새 ClassLabel 생성

    def remap(example):  # 라벨 인덱스를 새 인덱스로 재매핑
        name = train_ds.features["label"].int2str(example["label"])  # 기존 인덱스→이름
        example["label"] = names.index(name)  # 이름→새 인덱스
        return example

    train_ds = train_ds.map(remap, num_proc=os.cpu_count()//2,  # 병렬 맵 적용(캐시 사용)
                            load_from_cache_file=True, desc="Remap train")
    val_ds   = val_ds.map(remap,   num_proc=os.cpu_count()//2,
                            load_from_cache_file=True, desc="Remap val")

    train_ds = train_ds.cast_column("label", new_lbl)  # 라벨 스키마를 새 ClassLabel로 캐스팅
    val_ds   = val_ds.cast_column("label",  new_lbl)

    # 4) RGB 통일 (결정적)
    def to_rgb(example):  # 비-RGB 이미지를 RGB로 변환
        img = example["image"]  # PIL 이미지
        if img.mode != "RGB":  # 모드가 RGB가 아니면
            img = img.convert("RGB")  # RGB 변환
        example["image"] = img  # 교체
        return example

    train_ds = train_ds.map(to_rgb, num_proc=os.cpu_count()//2,  # RGB 통일(train)
                            load_from_cache_file=True, desc="RGB train")
    val_ds   = val_ds.map(to_rgb,   num_proc=os.cpu_count()//2,  # RGB 통일(val)
                            load_from_cache_file=True, desc="RGB val")

    # ✅ 여기서 끝! set_transform 안 씀
    return DatasetDict({"train": train_ds, "test": val_ds})  # 최종 DatasetDict 반환

In [ ]:
# PyTorch Dataset 래퍼
# --------------------------------------------------
class FruitHFDataset(Dataset):  # HF Dataset을 PyTorch Dataset으로 감싸는 래퍼
    def __init__(self, hf_dataset, transform=None):  # HF dataset과 transform 주입
        self.ds = hf_dataset  # 내부 참조
        self.tf = transform  # 변환 파이프라인

    def __len__(self):  # 전체 샘플 수
        return len(self.ds)

    def __getitem__(self, idx):  # 인덱스로 샘플 접근
        item = self.ds[idx]            # image: PIL.Image, label: int
        img  = item["image"]  # 이미지 추출
        if self.tf is not None:  # 변환이 설정되어 있으면
            img = self.tf(img)         # Tensor(C,H,W)로 변환/증강
        label = item["label"]  # 라벨 추출
        # long 보장
        import torch  # 지역 import (원본 코드 유지)
        if not torch.is_tensor(label):  # 라벨이 텐서가 아니면
            import torch  # 지역 import (원본 코드 유지)
            label = torch.tensor(label, dtype=torch.long)  # LongTensor로 변환
        else:
            label = label.to(dtype=torch.long)  # dtype 보정
        return img, label  # (이미지 텐서, 라벨 텐서) 반환

In [ ]:
# 모델 코드 cmt기반 cnn+t     
# ------------------------------------------------------------
import math
import torch
import torch.nn as nn

class DropPath(nn.Module):  # Stochastic Depth 구현
    """ per-sample DropPath (Stochastic Depth) """
    def __init__(self, drop_prob: float = 0.0):
        super().__init__()
        self.drop_prob = float(drop_prob)

    def forward(self, x):
        if self.drop_prob == 0.0 or not self.training:
            return x
        keep_prob = 1.0 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = x.new_empty(shape).bernoulli_(keep_prob)
        return x.div(keep_prob) * random_tensor

class DepthwiseConv2d(nn.Module):
    def __init__(self, channels, k=3, s=1, p=1, bias=False):
        super().__init__()
        self.dw = nn.Conv2d(channels, channels, k, s, p, groups=channels, bias=bias)

    def forward(self, x):
        return self.dw(x)

class ConvBNGELU(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, k, s, p, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch, eps=1e-5, momentum=0.1)
        self.act  = nn.GELU()

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class ConvStage(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.ds   = ConvBNGELU(in_ch, out_ch, k=3, s=2, p=1)
        self.body = ConvBNGELU(out_ch, out_ch, k=3, s=1, p=1)

    def forward(self, x):
        x = self.ds(x)
        x = self.body(x)
        return x
    
class LPU(nn.Module):
    """ Local Perception Unit: 3x3 depthwise → GELU → BN (채널 보존) """
    def __init__(self, channels):
        super().__init__()
        self.dw  = DepthwiseConv2d(channels, k=3, s=1, p=1, bias=False)
        self.bn  = nn.BatchNorm2d(channels, eps=1e-5, momentum=0.1)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.dw(x)
        x = self.bn(x)
        x = self.act(x)
        return x
    
class MLP(nn.Module):
    def __init__(self, dim, mlp_ratio=3.0, drop=0.1):
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.fc1 = nn.Linear(dim, hidden)
        self.act = nn.GELU()
        self.fc2 = nn.Linear(hidden, dim)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x); x = self.act(x); x = self.drop(x)
        x = self.fc2(x); x = self.drop(x)
        return x

# -----------------------------------------------------------
# [수정됨] LPU가 포함된 TransformerBlock
# -----------------------------------------------------------
class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=3.0, attn_drop=0.0, proj_drop=0.1, drop_path=0.0): 
        super().__init__()
        # 1. LPU (Local Perception Unit) 내부 탑재
        self.lpu = LPU(dim)

        self.norm1 = nn.LayerNorm(dim, eps=1e-6)
        self.attn  = nn.MultiheadAttention(dim, num_heads, dropout=attn_drop, batch_first=True)
        self.drop1 = nn.Dropout(proj_drop)
        self.dp1   = DropPath(drop_path)

        self.norm2 = nn.LayerNorm(dim, eps=1e-6)
        self.mlp   = MLP(dim, mlp_ratio=mlp_ratio, drop=proj_drop)
        self.dp2   = DropPath(drop_path)

    def forward(self, x):
        # x: (B, N, C) - Token 형태
        B, N, C = x.shape
        
        # --- [1] LPU 적용 (Token -> Image -> LPU -> Token) ---
        # 정사각형 이미지라고 가정: H = W = sqrt(N)
        H = W = int(math.sqrt(N)) 
        
        x_reshaped = x.transpose(1, 2).view(B, C, H, W) # (B, C, H, W)
        x_reshaped = self.lpu(x_reshaped)               # LPU 통과
        x_lpu = x_reshaped.flatten(2).transpose(1, 2)   # 다시 (B, N, C)
        
        x = x + x_lpu # Residual Connection (논문 구현에 따라 직렬 연결 혹은 잔차 연결)

        # --- [2] Self-Attention ---
        y, _ = self.attn(self.norm1(x), self.norm1(x), self.norm1(x), need_weights=False)
        x = x + self.dp1(self.drop1(y))

        # --- [3] MLP (FFN) ---
        x = x + self.dp2(self.mlp(self.norm2(x)))
        return x
# ------------------------------------------------------------
# [추가됨] EMA (Exponential Moving Average) 클래스
# ------------------------------------------------------------
from copy import deepcopy

class ModelEma(nn.Module):
    def __init__(self, model, decay=0.9999, device=None):
        super().__init__()
        # 매개변수 복사 (requires_grad=False)
        self.module = deepcopy(model)
        self.module.eval()
        self.decay = decay
        self.device = device
        if self.device is not None:
            self.module.to(device=device)

    def _update(self, model, update_fn):
        with torch.no_grad():
            for ema_v, model_v in zip(self.module.state_dict().values(), model.state_dict().values()):
                if self.device is not None:
                    model_v = model_v.to(device=self.device)
                ema_v.copy_(update_fn(ema_v, model_v))

    def update(self, model):
        self._update(model, update_fn=lambda e, m: self.decay * e + (1. - self.decay) * m)

    def set(self, model):
        self._update(model, update_fn=lambda e, m: m)
# -----------------------------------------------------------
# [수정됨] 외부 LPU 제거 및 내부 로직 변경된 CMTClassifier
# -----------------------------------------------------------
class CMTClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        stem_channels: int = 64,
        c_stage1: int = 96,
        c_stage2: int = 128,
        c_stage3: int = 160,
        t_dim1: int = 256,  t_heads1: int = 4,  t_depth1: int = 3,  t_mlp1: float = 3.0,
        t_dim2: int = 384,  t_heads2: int = 6,  t_depth2: int = 6,  t_mlp2: float = 3.5,
        attn_drop: float = 0.0,
        proj_drop: float = 0.1,
        drop_path_rate: float = 0.0,
    ):
        super().__init__()
        self.num_classes = num_classes

        # ----- CNN stem (224 -> 112) -----
        self.stem = nn.Sequential(
            ConvBNGELU(3, stem_channels // 2, k=3, s=2, p=1),
            ConvBNGELU(stem_channels // 2, stem_channels, k=3, s=1, p=1),
        )

        # ----- CNN stages (112 -> 56 -> 28 -> 14) -----
        self.stage1 = ConvStage(stem_channels, c_stage1)
        self.stage2 = ConvStage(c_stage1, c_stage2)
        self.stage3 = ConvStage(c_stage2, c_stage3)

        # ----- to embed (14x14, C3 -> D1) -----
        self.to_embed1 = nn.Conv2d(c_stage3, t_dim1, kernel_size=1, stride=1, padding=0, bias=True)

        # ----- Stage A @14x14 : Transformer(depth=t_depth1) -----
        # [삭제됨] self.lpu1 = LPU(t_dim1) <- 이제 블록 안에 있음
        
        dpr1 = torch.linspace(0, drop_path_rate * 0.5, steps=t_depth1).tolist()
        self.trans1 = nn.Sequential(*[
            TransformerBlock(
                dim=t_dim1, num_heads=t_heads1, mlp_ratio=t_mlp1,
                attn_drop=attn_drop, proj_drop=proj_drop, drop_path=dpr1[i]
            ) for i in range(t_depth1)
        ])

        # ----- down tokens: 14->7, D1->D2 -----
        self.down_tokens = nn.Sequential(
            nn.Conv2d(t_dim1, t_dim2, kernel_size=3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(t_dim2, eps=1e-5, momentum=0.1),
            nn.GELU(),
        )

        # ----- Stage B @7x7 : Transformer(depth=t_depth2) -----
        # [삭제됨] self.lpu2 = LPU(t_dim2) <- 이제 블록 안에 있음

        dpr2 = torch.linspace(drop_path_rate * 0.5, drop_path_rate, steps=t_depth2).tolist()
        self.trans2 = nn.Sequential(*[
            TransformerBlock(
                dim=t_dim2, num_heads=t_heads2, mlp_ratio=t_mlp2,
                attn_drop=attn_drop, proj_drop=proj_drop, drop_path=dpr2[i]
            ) for i in range(t_depth2)
        ])

        # ----- Head -----
        self.head_norm = nn.LayerNorm(t_dim2, eps=1e-6)
        self.fc        = nn.Linear(t_dim2, num_classes)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm2d)):
            if hasattr(m, "weight") and m.weight is not None: nn.init.ones_(m.weight)
            if hasattr(m, "bias") and m.bias is not None:     nn.init.zeros_(m.bias)

    def forward(self, x):
        # CNN 얕은 특징
        x = self.stem(x)          # 224 -> 112
        x = self.stage1(x)        # 112 -> 56
        x = self.stage2(x)        # 56  -> 28
        x = self.stage3(x)        # 28  -> 14

        # 임베딩 (채널 C3 -> T_dim1)
        x = self.to_embed1(x)     # B x D1 x 14 x 14

        # [수정됨] Stage A: Flatten 먼저 -> Transformer (내부에서 LPU 처리)
        x = x.flatten(2).transpose(1, 2)           # B x 196 x D1 (패치 토큰화)
        x = self.trans1(x)                         # Transformer (내부에 LPU 포함됨)
        
        # Downsample을 위해 다시 이미지 형태로 복구
        B, N, C = x.shape
        H = W = int(math.sqrt(N)) # 14
        x = x.transpose(1, 2).view(B, C, H, W)     # 다시 (B, D1, 14, 14)

        # Downsample tokens: 14 -> 7
        x = self.down_tokens(x)                    # B x D2 x 7 x 7

        # [수정됨] Stage B: Flatten 먼저 -> Transformer (내부에서 LPU 처리)
        x = x.flatten(2).transpose(1, 2)           # B x 49 x D2
        x = self.trans2(x)                         # Transformer (내부에 LPU 포함됨)

        # Head
        x = x.mean(dim=1)                          # GAP
        x = self.head_norm(x)
        logits = self.fc(x)
        return logits
# ------------------------------------------------------------

In [ ]:
# 손실함수 클래스


class FocalLoss(nn.Module):  # 멀티클래스 Focal Loss 구현
    """
    Multi-class Focal Loss with per-class alpha.
    - inputs: logits (B, C)
    - targets: int labels (B,)
    """
    def __init__(self, alpha=None, gamma=2.0, reduction="mean", eps=1e-8):  # 하이퍼파라미터
        super().__init__()
        self.gamma = float(gamma)  # 난이도 조절 지수
        self.reduction = reduction  # 리덕션 방식
        self.eps = float(eps)  # 수치 안정성 epsilon

        if alpha is not None:  # 클래스 가중치가 주어지면
            alpha = torch.as_tensor(alpha, dtype=torch.float32)  # 텐서화
        self.register_buffer("alpha", alpha if alpha is not None else None)  # 버퍼로 등록(디바이스 이동 자동)

    def forward(self, inputs, targets):  # 입력 로짓, 정답 라벨
        # logits -> log-prob/prob
        log_probs = F.log_softmax(inputs, dim=1)   # (B, C) 로그소프트맥스
        probs     = log_probs.exp()                # (B, C) 확률

        targets = targets.long()  # 라벨 long 보장
        log_pt  = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)  # 정답 클래스 로그확률
        pt      = probs.gather(1, targets.unsqueeze(1)).squeeze(1)      # 정답 클래스 확률

        # --- 수치 안정화 ---
        pt = pt.clamp(min=self.eps, max=1. - self.eps)  # 확률 클램프
        # log_pt는 log_softmax 결과라 이미 안정적이지만, 혹시 모를 NaN 방지:
        log_pt = torch.log(pt)  # 로그 재계산

        # alpha_t
        if self.alpha is not None:  # 클래스별 가중치가 있으면
            alpha_t = self.alpha[targets]  # (B,)
        else:
            alpha_t = torch.ones_like(pt)  # 없으면 균등 가중

        # focal term
        focal = (1.0 - pt).pow(self.gamma)  # 어려운 샘플 가중↑
        loss  = -alpha_t * focal * log_pt  # Focal loss 공식

        if self.reduction == "mean":  # 평균 리덕션
            return loss.mean()
        elif self.reduction == "sum":  # 합 리덕션
            return loss.sum()
        return loss  # 리덕션 없음
# ------------------------------------------------------------
def mixup_data(x, y, alpha=0.2):
    """이미지 x, 라벨 y에 mixup 적용"""
    if alpha <= 0:
        return x, y, y, 1.0  # mixup 안 쓰는 경우 대비

    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam
def load_fold_models(num_folds, num_classes, device, ckpt_dir):
    models = []
    for fold in range(1, num_folds + 1):
        m = CMTClassifier(num_classes).to(device)
        path = os.path.join(ckpt_dir, f"best_model_fold{fold}.pt")
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()
        models.append(m)
    return models

@torch.inference_mode()
def ensemble_logits(models, x):
    # logits 평균(softmax 전에 평균내는 방식)
    logits_sum = 0
    for m in models:
        logits_sum = logits_sum + m(x)
    return logits_sum / len(models)
@torch.inference_mode()
def ensemble_logits_tta_hflip(models, x):
    # x: (N,C,H,W)
    x_flip = torch.flip(x, dims=[3])  # width 축 flip
    logits = ensemble_logits(models, x)
    logits_flip = ensemble_logits(models, x_flip)
    return (logits + logits_flip) / 2


In [ ]:
# 메인
def main():  # 전체 파이프라인 실행 함수
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("device:", device)
    if torch.cuda.is_available():
        print(torch.cuda.get_device_name(0))

    final_dataset = prepare_dataset()
    names = final_dataset["train"].features["label"].names
    save_dir = "C:/Users/user/Desktop/deep/model_data"
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "label_names.json"), "w", encoding="utf-8") as f:
        json.dump(names, f, ensure_ascii=False)

    num_classes = len(final_dataset["train"].features["label"].names)

    # ----- class-balanced alpha (FocalLoss용) -----
    train_labels = [int(x) for x in final_dataset["train"]["label"]]
    counts = Counter(train_labels)
    class_counts = [counts[i] for i in range(num_classes)]
    beta = 0.999
    effective_num = [1.0 - (beta ** c) for c in class_counts]
    raw_alpha = torch.tensor([(1.0 - beta) / (en if en > 0 else 1e-8) for en in effective_num], dtype=torch.float32)
    alpha = (raw_alpha / raw_alpha.sum()) * num_classes
    print("alpha:", alpha.tolist())


    # ================= [설정 변경] =================
    EPOCHS = 120
    # 마지막 5 epoch는 파인튜닝 (Cool-down)
    FINETUNE_EPOCHS = 20
    
    BATCH_SIZE = 192
    K = 3
    
    # Mixup 설정
    MIXUP_ALPHA = 0.8  # Mixup 강도 (0.08 -> 0.8로 약간 높임, 데이터가 적으면 강한게 좋음)
    MIXUP_P     = 0.5
    
    # 학습률
    LR_CNN = 5e-5
    LR_TRANS = 1e-4
    WEIGHT_DECAY = 1e-4
    
    # [전략] EMA Decay 설정
    EMA_DECAY = 0.999  
    
    USE_CE_LS = True
    LABEL_SMOOTHING = 0.01

    # [전략] 파인튜닝용 약한 증강 (ResizeCrop + Flip만 하고, ColorJitter/Erasing 제거)
    ft_transform = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.9, 1.0), ratio=(3/4, 4/3)), # 스케일 크게
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
    ])
    # ===============================================

    labels  = np.asarray(final_dataset["train"]["label"], dtype=np.int64)
    indices = np.arange(len(final_dataset["train"]), dtype=np.int64)
    skf = StratifiedKFold(n_splits=K, shuffle=True, random_state=42)

    fold_accs = []
    start_time = time.time()
    histories = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(indices, labels), 1):
        best_acc_fold = 0.0
        print(f"\n================ Fold {fold}/{K} 시작 ================")
        fold_start = time.time()
            
        history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

        train_split = final_dataset["train"].select(list(train_idx))
        val_split   = final_dataset["train"].select(list(val_idx))

        # 데이터셋 생성
        train_ds = FruitHFDataset(train_split, transform=train_transform)
        val_ds   = FruitHFDataset(val_split,  transform=val_transform)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
        
        torch.backends.cudnn.benchmark = True

        # --- 모델 초기화 ---
        model = CMTClassifier(num_classes).to(device)
        
        # [전략] EMA 모델 초기화
        ema = ModelEma(model, decay=EMA_DECAY, device=device)

        if USE_CE_LS:
            criterion = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING).to(device)
        else:
            criterion = FocalLoss(alpha=alpha.to(device), gamma=2.0).to(device)

        # 파라미터 그룹
        cnn_modules = [model.stem, model.stage1, model.stage2, model.stage3]
        trans_modules = [model.to_embed1, model.trans1, model.down_tokens, model.trans2, model.head_norm, model.fc]

        cnn_params = []
        trans_params = []
        for m in cnn_modules: cnn_params += list(m.parameters())
        for m in trans_modules: trans_params += list(m.parameters())

        optimizer = torch.optim.AdamW([
            {"params": cnn_params,  "lr": LR_CNN},
            {"params": trans_params, "lr": LR_TRANS},
        ], weight_decay=WEIGHT_DECAY)

        from torch.optim.lr_scheduler import CosineAnnealingLR
        # 스케줄러는 전체 Epoch 기준 (파인튜닝 구간에서도 LR이 줄어들도록 자연스럽게)
        scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
        scaler = GradScaler()

        val_acc_list = []
        val_loss_list = []
        val_f1_list   = []

        for epoch in range(1, EPOCHS+1):
            epoch_start = time.time()
            
            # ================= [전략 1] 파인튜닝 (Cool-down) 체크 =================
            is_finetuning = (epoch > EPOCHS - FINETUNE_EPOCHS)
            
            if is_finetuning:
                print(f"▶ Fold {fold} | Epoch {epoch} [Fine-tuning Mode! Mixup OFF, Weak Aug]")
                # 1. 증강 교체 (강한 증강 -> 약한 증강)
                train_ds.tf = ft_transform 
                # 2. Mixup OFF (아래 루프에서 처리)
            else:
                print(f"\n▶ Fold {fold} | Epoch {epoch}/{EPOCHS}")
            # ======================================================================

            # ---- [Train] ----
            model.train()
            total, correct, loss_sum = 0, 0, 0.0
            pbar = tqdm(train_loader, desc=f"Fold {fold} Epoch {epoch} [Train]", ncols=100)

            for x, y in pbar:
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                # [전략] 파인튜닝 때는 Mixup 끄기
                if is_finetuning:
                    do_mix = False
                else:
                    do_mix = (np.random.rand() < MIXUP_P)

                if do_mix:
                    x_in, y_a, y_b, lam = mixup_data(x, y, alpha=MIXUP_ALPHA)
                else:
                    x_in, y_a, y_b, lam = x, y, y, 1.0

                with autocast("cuda"):
                    out = model(x_in)
                    if do_mix:
                        loss = lam * criterion(out, y_a) + (1 - lam) * criterion(out, y_b)
                    else:
                        loss = criterion(out, y)

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                # [전략 2] EMA 업데이트 (학습 스텝마다)
                ema.update(model)

                bs = x.size(0)
                loss_sum += loss.item() * bs
                pred = out.argmax(1)
                if do_mix:
                    correct += (lam * (pred == y_a).float() + (1 - lam) * (pred == y_b).float()).sum().item()
                else:
                    correct += (pred == y).sum().item()
                total += bs

            tr_acc  = correct / max(1, total)
            tr_loss = loss_sum / max(1, total)
            print(f"Train ▶ acc: {tr_acc:.4f} | loss: {tr_loss:.4f}")

            # ---- [Validation] ----
            # [전략] 검증 시 EMA 모델 사용 (성능이 더 안정적임)
            # 주의: EMA 모델은 eval 모드이므로 model.eval() 불필요하지만 명시적으로 둠
            val_model = ema.module 
            val_model.eval()
            
            v_total, v_correct, v_loss_sum = 0, 0, 0.0
            all_preds, all_labels, all_logits = [], [], []

            with torch.inference_mode():
                for x, y in tqdm(val_loader, desc=f"Fold {fold} Epoch {epoch} [Val]", ncols=100):
                    x = x.to(device, non_blocking=True)
                    y = y.to(device, non_blocking=True)

                    with autocast("cuda"):
                        # [전략 3] TTA (Horizontal Flip) 적용 - 검증부터 적용해서 성능 확인
                        out  = (val_model(x) + val_model(torch.flip(x, dims=[3]))) / 2
                        v_loss = criterion(out, y)

                    bs = x.size(0)
                    v_loss_sum += v_loss.item() * bs
                    preds = out.argmax(1)
                    v_correct += (preds == y).sum().item()
                    v_total   += bs

                    all_preds.extend(preds.detach().cpu().numpy())
                    all_labels.extend(y.detach().cpu().numpy())
                    all_logits.append(out.detach().cpu().numpy())

            va_acc  = v_correct / max(1, v_total)
            va_loss = v_loss_sum / max(1, v_total)
            
            history["train_loss"].append(tr_loss)
            history["train_acc"].append(tr_acc)
            history["val_loss"].append(va_loss)
            history["val_acc"].append(va_acc)
            
            all_logits = np.concatenate(all_logits, axis=0)
            va_f1   = f1_score(all_labels, all_preds, average="macro")
            va_bal  = balanced_accuracy_score(all_labels, all_preds)

            try:
                va_top2 = top_k_accuracy_score(all_labels, all_logits, k=2, labels=np.arange(all_logits.shape[1]))
                va_top3 = top_k_accuracy_score(all_labels, all_logits, k=3, labels=np.arange(all_logits.shape[1]))
            except:
                va_top2 = va_top3 = None

            val_acc_list.append(va_acc)
            val_loss_list.append(va_loss)
            val_f1_list.append(va_f1)
            print(f"Val (EMA) ▶ acc: {va_acc:.4f} | f1: {va_f1:.4f} | loss: {va_loss:.4f}")
            
            # 저장
            if va_acc > best_acc_fold + 1e-6:
                best_acc_fold = va_acc
                save_path = os.path.join(save_dir, f"best_model_fold{fold}.pt")
                # [전략] EMA 모델의 가중치를 저장
                torch.save(ema.module.state_dict(), save_path)
                print(f"New best model (EMA) saved! (fold={fold}, acc={best_acc_fold:.4f})")

            epoch_time = time.time() - epoch_start
            print(f"Epoch {epoch} 완료 (소요시간: {epoch_time:.2f}초)")

            scheduler.step()

        # --- Fold 종료 ---
        fold_time = time.time() - fold_start
        histories.append(history)
        print(f"✅ Fold {fold} 완료! (소요시간: {fold_time/60:.2f}분)")
        fold_accs.append(val_acc_list)

    # --- 전체 요약 ---
    ckpt_dir = save_dir
    models = load_fold_models(K, num_classes, device, ckpt_dir)

    test_ds = FruitHFDataset(final_dataset["test"], transform=val_transform)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    t_total = t_correct = 0
    print("\n[최종 평가] Holdout Test Set (Ensemble + TTA)")
    
    # [전략] TTA + 앙상블 적용
    for x, y in tqdm(test_loader, ncols=100):
        x = x.to(device)
        y = y.to(device)
        with autocast("cuda"):
            logits = ensemble_logits_tta_hflip(models, x)
        pred = logits.argmax(1)
        t_correct += (pred == y).sum().item()
        t_total += y.size(0)

    print("Final Holdout Acc:", t_correct / t_total)
    
    total_time = time.time() - start_time
    print(f"\n================ 학습종료 총 소요시간: {total_time/60:.2f}분 ================")

    # 그래프 그리기
    epochs = range(1, EPOCHS + 1)
    plt.figure(figsize=(10, 4))
    plt.subplot(1, 2, 1)
    # 마지막 Fold의 history만 그림
    plt.plot(epochs, history["train_loss"], label="train_loss")
    plt.plot(epochs, history["val_loss"],   label="val_loss")
    plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Loss (Last Fold)"); plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history["train_acc"], label="train_acc")
    plt.plot(epochs, history["val_acc"],   label="val_acc")
    plt.xlabel("Epoch"); plt.ylabel("Acc"); plt.title("Accuracy (Last Fold)"); plt.legend()
    plt.tight_layout()
    plt.show()

    torch.save(model.state_dict(), "last_model_weights.pt")

if __name__ == "__main__":
    main()